<a href="https://colab.research.google.com/github/Sinrez/PythonProjects/blob/main/ECDH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Объяснение тут https://t.me/nkodw/3520

In [1]:
class EllipticCurve:
    """
    Механика формирования общего секрета через
    эллиптическую кривую ECDH (Elliptic Curve Diffie-Hellman)
    """
    def __init__(self, a, b, p):
        #конструктор, параметры кривой y^2 = x^3 + a*x + b (mod p)
        self.a = a
        self.b = b
        self.p = p

    def is_on_curve(self, point):
        """Проверка, лежит ли точка на кривой"""
        if point is None: return True
        x, y = point
        return (y**2 - (x**3 + self.a * x + self.b)) % self.p == 0

    def add_points(self, p1, p2):
        """сложение точек на кривой"""
        if p1 is None: return p2 # P + O = P
        if p2 is None: return p1 # O + P = P

        x1, y1 = p1
        x2, y2 = p2

        if x1 == x2 and y1 != y2: return None # None — это нейтральный элемент (точка на бесконечности)

        if x1 == x2:
            m = (3 * x1**2 + self.a) * pow(2 * y1, -1, self.p) #производная
        else:
            m = (y2 - y1) * pow(x2 - x1, -1, self.p) # Сложение разных точек, секущая, наклон кривой

        x3 = (m**2 - x1 - x2) % self.p
        y3 = (m * (x1 - x3) - y1) % self.p
        return (x3, y3)

    def multiply_point(self, k, point):
        result = None
        addend = point
        while k:
            if k & 1:
                result = self.add_points(result, addend)
            addend = self.add_points(addend, addend)
            k >>= 1 #побитовый сдвиг вправо, умножение
        return result

# Инициализация кривой
# y^2 = x^3 + 2x + 2 (mod 17)
curve = EllipticCurve(2, 2, 17)
G = (5, 1)  # Базовая точка

# секрет Алисы
priv_a = 3
pub_a = curve.multiply_point(priv_a, G)

# секрет Боба
priv_b = 5
pub_b = curve.multiply_point(priv_b, G)

# Общий секрет
shared_a = curve.multiply_point(priv_a, pub_b)
shared_b = curve.multiply_point(priv_b, pub_a)

print(f"Публичный ключ Алисы: {pub_a}")
print(f"Публичный ключ Боба: {pub_b}")
print(f"Общий секрет (Алиса): {shared_a}")
print(f"Общий секрет (Боб): {shared_b}")
print(f"Ключи совпали? {'Да!' if shared_a == shared_b else 'Нет!'}")

Публичный ключ Алисы: (10, 6)
Публичный ключ Боба: (9, 16)
Общий секрет (Алиса): (3, 16)
Общий секрет (Боб): (3, 16)
Ключи совпали? Да!
